In [1]:
from iFinDPy import *
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
from datetime import date
import pandas as pd
import numpy as np

e:\ProgramData\Anaconda3\envs\QGIBG\Lib\site-packages\iFinDPy.pth


In [2]:
'''host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)'''

"host='localhost'\nuser='Local_Editor'\npassword='QuantumGalaxy'\ndatabase='qgdbs'\nmysql=MYSQL(host,user,password,database)"

In [3]:
def query_code(mysql,region,today):
    '''return a list of df for every ticker in spcific region, df[code,close] [today-370,today]'''
   
    today=date.today()-timedelta(1)
    start_date=(today-timedelta(370)).__format__('%Y-%m-%d')
    sql="select a.code, b.date,b.close_price from ticker_info a inner join ticker_data b on a.code = b.code where region ='%s' and b.date>'%s'"%(region,start_date)
    result=mysql.read_query(sql)
    result=list(result)
    df=pd.DataFrame(result,columns=['code','date','close'])
    group=df.groupby('code') #tuple(code,df)
    df_list=[]
    #result=pd.DataFrame({code: data["close"] for code, data in group})
    result
    for t in group:
        
        t[1]['date']=pd.to_datetime(t[1]['date'])
        df=t[1].set_index('date')
        df1=pd.DataFrame(data={df.iloc[1]['code']:df['close']},index=df.index)
        df_list.append(df1)
    result=pd.concat(df_list,axis=1)
    result.replace({None:np.nan},inplace=True)
    result.fillna(method='pad',limit=10,inplace=True)
    result=result.astype('float')
    return result



In [4]:
def compute_chg_from_days(df,today:datetime.date,days:int):
    try:# for some new ticker which has no date before the day
        df=df.truncate(after=today)
        #print(days)
        former=df.truncate(after=today-timedelta(days)).iloc[-1]
        
        return (df.iloc[-1]-former)/former
    except:
        return None

    
    

In [5]:
def compute_std_chg_from_days(df,today,days):
    #days=7,30,90
    try:
        df=df.truncate(after=today)
        after=np.log(df.iloc[-1])
        former=np.log(df.truncate(after=today-timedelta(days)).iloc[-1])
        stdchg60=(np.log(df)-np.log(df.shift(1))).iloc[-60:,:].std()
        
        result=(after-former)/(np.sqrt(days)*stdchg60)
        return result
    except:

        raise 

In [8]:
def compute_std_chg_from_days(df,today,days):
    #days=7,30,90
    try:
        df=np.log(df)
        df=df.truncate(after=today)
        after=df.iloc[-1]
        former=df.truncate(after=today-timedelta(days)).iloc[-1]
        stdchg60=(df-df.shift(1)).iloc[-60:,:].std()
        
        result=(after-former)/(np.sqrt(days)*stdchg60)
        return result
    except:

        raise 

In [6]:
def compute_stdchg60(df,today):
    try:
        df=np.log(df)
        df=df.truncate(after=today)
        stdchg60=(df-df.shift(1)).iloc[-60:,:].std()
        

        return stdchg60
    except:

        raise 

In [9]:
def compute(df, today: datetime.date):
    processed_df = pd.DataFrame(
        data={
            'date': today,
            'chg1d': compute_chg_from_days(df, today, 1),
            'chg3d': compute_chg_from_days(df, today, 3),
            'chg2w': compute_chg_from_days(df, today, 14),
            'chg1m': compute_chg_from_days(df, today, 30),
            'chg3m': compute_chg_from_days(df, today, 91),
            'chg6m': compute_chg_from_days(df, today, 182),
            'chg1y': compute_chg_from_days(df, today, 365),
            'stdchg7d': compute_std_chg_from_days(df, today, 7),
            'stdchg1m': compute_std_chg_from_days(df, today, 30),
            'stdchg3m': compute_std_chg_from_days(df, today, 90),
            'stdchg': compute_stdchg60(df, today)
        },
        index=df.columns,
    )
    #processed_df['date'],processed_df['chg1d']=today,compute_chg_from_days(test,today,1)
    #chg1d,chg3d,chg2w,chg1m,chg3m,chg6m,chg1y=compute_chg_from_days(df,today,1),compute_chg_from_days(df,today,3),compute_chg_from_days(df,today,14),compute_chg_from_days(df,today,30),compute_chg_from_days(df,today,90),compute_chg_from_days(df,today,180),compute_chg_from_days(df,today,365)
    #stdchg7d,stdchg1m,stdchg3m=compute_std_chg_from_days(df,today,7),compute_std_chg_from_days(df,today,30),compute_std_chg_from_days(df,today,90)
    #processed_df=pd.DataFrame(data=[[today,chg1d,chg3d,chg2w,chg1m,chg3m,chg6m,chg1y,stdchg7d,stdchg1m,stdchg3m]],columns=['date','chg1d','chg3d','chg2w','chg1m','chg3m','chg6m','chg1y','stdchg7d','stdchg1m','stdchg3m'])
    return processed_df

In [10]:
def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/processed_data_log.log",'a')
    fh.setLevel(logging.INFO)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger

In [11]:
logger=get_logger()

In [12]:
def job(region):
    logger.info('begin job for %s'%region)
    THS_iFinDLogin('lzxh0011','450503')
    host='localhost'
    user='Local_Editor'
    password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(host,user,password,database)

    if region=='US':
        today=date.today()-timedelta(1)
    else:
        today=date.today()

    df=query_code(mysql,region,today)
    logger.info('begin compute')
    result=compute(df,today)
    result=result.reset_index().rename(columns={'index':'code'})
    result.replace([np.inf, -np.inf], np.nan)
    result.replace({np.nan: None},inplace=True)
    sql='insert ignore into processed_data (code,date,change_rate_1day,change_rate_3day,change_rate_2week,change_rate_1month,change_rate_3month,change_rate_6month,change_rate_1year,stdchg7d,stdchg1m,stdchg3m,stdchg) values(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'
    val=result.values.tolist()
    number=mysql.wirte_many_query(sql,val)
    return result
        
        

In [13]:
#result=job('cn')

result=job('hk')


05-26 21:42 logger       INFO     begin job for hk
05-26 21:42 logger       INFO     using username Local_Editor
05-26 21:42 logger       INFO     accessed to database <qgdbs>
05-26 21:42 logger       INFO     got a cursor
05-26 21:42 logger       INFO     Accepted read sql query "select a.code, b.date,b.close_price from ticker_info a inner join ticker_data b on a.code = b.code where region ='hk' and b.date>'2021-05-20'"
05-26 21:42 logger       INFO     begin compute
e:\ProgramData\Anaconda3\envs\QGIBG\lib\site-packages\pandas\core\internals\blocks.py:402: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
05-26 21:42 logger       INFO     got a cursor
05-26 21:42 logger       INFO     executed and commited 2585 sql "insert ignore into processed_data (code,date,change_rate_1day,change_rate_3day,change_rate_2week,change_rate_1month,change_rate_3month,change_rate_6month,change_rate_1year,stdchg7d,stdchg1m,stdchg3m,stdchg) values(%s,%s,%s,%s,%s,%s,%s

In [200]:

sql1='insert ignore into processed_data (code,date,change_rate_1day,change_rate_3day,change_rate_2week,change_rate_1month,change_rate_3month,change_rate_6month,change_rate_1year,stdchg7d,stdchg1m,stdchg3m,stdchg) values(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)'
val=result.values.tolist()
result1=mysql.wirte_many_query(sql1,val)
result1

0

In [198]:
result.replace([np.inf, -np.inf], np.nan,inplace=True)

result.replace({np.nan: None},inplace=True)